# Amaliyot 2. Maʼlumotlar toʻplamini tayyorlash

2-interaktiv laboratoriyaga («Maʼlumotlar toʻplamini tayyorlash») hamroh. Kurs mavzusi: tanlanmalar shakllantirish, belgilash va maʼlumotlar sifati

Daftar oʻz-oʻzidan yetarli: birinchi katak `common.py` faylini oʻquv korpusi va yordamchi funksiyalar bilan yaratadi. Kataklarni yuqoridan pastga ishga tushiring. Korpus oʻquv (shablon) boʻlib, haqiqiy masalada oʻz CSV faylingizni quyidagi ustunlar bilan qoʻying: `text` va `label`.

In [ ]:
%%writefile common.py
"""Matnlarni tasniflash va klasterlash amaliyotlari uchun umumiy funksiyalar."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "sport": ("oʻyin jamoa gol oʻyinchi murabbiy chempionat turnir gʻalaba magʻlubiyat mavsum stadion muxlis hakam hisob "
              "himoyachi hujumchi darvozabon kubok liga final transfer klub zarba penalti musobaqa sportchi "
              "rekord medal olimpiada mashq durang").split(),
    "siyosat": ("qonun deputat saylov partiya ovoz prezident vazir hukumat parlament islohot byudjet "
                "sanksiya nomzod koalitsiya vazirlik farmon tuzatish loyiha muxolifat referendum diplomatiya "
                "hokim konstitutsiya majlis komissiya muzokara sammit saylovchi mandat vakolat").split(),
    "oshxona": ("shoʻrva retsept tuz goʻsht pishirmoq duxovka sabzavot tova qozon xamir shakar ziravor qaynatmoq qovurmoq "
                "dimlamoq sous bulyon salat oshpaz taom masalliq yogʻ un tuxum sut sarimsoq piyoz garmdori "
                "kartoshka shirinlik pishiriq marinad").split(),
    "texnologiya": ("protsessor dastur ilova smartfon neyrotarmoq algoritm maʼlumot server yangilanish dasturchi "
                    "kompyuter xotira ekran kamera brauzer tarmoq internet bulut kod platforma foydalanuvchi qurilma "
                    "chip batareya noutbuk dasturiy videokarta tizim interfeys shifrlash").split(),
}
NEUTRAL = ("bugun juda yangi aytishadi keyin ham mumkin edi boʻldi maʼlum qilindi kecha yaqinda hali hammasi "
           "bir nechta muhim birinchi").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """4 mavzudagi qisqa «hujjatlar»dan iborat oʻquv korpusi.
    noise: oʻz mavzusidan boʻlmagan soʻzlar ulushi (neytral yoki begona). noise qancha katta boʻlsa, masala shuncha qiyin."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """TF-IDF siyrak matritsasini tekislikka proyeksiyalash (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Har bir sinf uchun eng katta vaznli soʻzlarni chiqaradi (chiziqli modellar)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))


In [ ]:
from common import *

# Oʻquv korpusi: 4 ta mavzu. noise — "begona" soʻzlar ulushi (qancha koʻp boʻlsa, shuncha qiyin).
df = make_corpus(n_per_class=150, noise=0.5, seed=0)
# Oʻquv korpusi oʻrniga oʻzingiznikini qoʻying: df = pd.read_csv("texts.csv")  # ustunlar: text, label
print(df.label.value_counts().to_dict())
df.head()

## 1. train / val / test boʻlish, stratifikatsiya bilan

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score, accuracy_score

train_val, test = train_test_split(df, test_size=0.2, stratify=df.label, random_state=0)
train, val = train_test_split(train_val, test_size=0.25, stratify=train_val.label, random_state=0)
print(len(train), len(val), len(test))
pd.DataFrame({"train": train.label.value_counts(), "val": val.label.value_counts(), "test": test.label.value_counts()})

Agar maʼlumotlar sinflar boʻyicha tartiblangan va aralashtirish yoʻq boʻlsa, test bitta sinfdan iborat boʻladi:

In [ ]:
srt = df.sort_values("label").reset_index(drop=True)
cut = int(len(srt) * 0.8)
print("Aralashtirmasdan test:", srt.label[cut:].value_counts().to_dict())

## 2. Maʼlumotlar sizib chiqishi
Testdagi oʻquv matnlari nusxalari bahoni oshirib yuboradi.

In [ ]:
vec = TfidfVectorizer().fit(train.text)
def acc_1nn(tr, te):
    m = KNeighborsClassifier(1, metric="cosine").fit(vec.transform(tr.text), tr.label)
    return accuracy_score(te.label, m.predict(vec.transform(te.text)))
leak = train.sample(len(test) // 2, random_state=0)
test_leak = pd.concat([test.iloc[: len(test) // 2], leak])
print("Halol test:", round(acc_1nn(train, test), 3), "| sizib chiqishli test:", round(acc_1nn(train, test_leak), 3))
print("Train/test umumiy matnlari:", len(set(train.text) & set(test.text)), "| sizib chiqish bilan:", len(set(train.text) & set(test_leak.text)))

## 3. Belgilash sifati: belgilovchilar kelishuvi (Koen κ)

In [ ]:
rng = np.random.default_rng(0)
classes = np.array(sorted(df.label.unique()))
def annotate(labels, p):
    labels = np.array(labels).copy()
    flip = rng.random(len(labels)) < p
    labels[flip] = [rng.choice(classes[classes != l]) for l in labels[flip]]
    return labels
Xtr, Xte = vec.transform(train.text), vec.transform(test.text)
rows = []
for p in [0, 0.1, 0.2, 0.3, 0.4]:
    a, b = annotate(train.label, p), annotate(train.label, p)
    clf = LogisticRegression(max_iter=1000).fit(Xtr, a)
    rows.append((p, cohen_kappa_score(a, b), accuracy_score(test.label, clf.predict(Xte))))
res = pd.DataFrame(rows, columns=["belgilovchi xatolari ulushi", "ikki belgilovchi kappa", "toza testda accuracy"])
display(res.round(3))
res.plot(x="belgilovchi xatolari ulushi", y=["ikki belgilovchi kappa", "toza testda accuracy"], marker="o")
plt.show()

## Topshiriqlar

1. Korpusda aniq dublikatlarni toping (`df.duplicated`) va boʻlishdan oldin olib tashlang. Nima uchun buni keyin emas, oldin qilish kerak?
2. Boʻlishni quring, bunda `stratify` ishlatmang va juda kichik kam uchraydigan sinf qoldiring (10 ta hujjat). Testda kam uchraydigan sinfga nima boʻladi?
3. Belgilovchilar uchun noaniq matnlarda κ ni oshiradigan yoʻriqnoma qoidalarini oʻylab toping.